# enVector KMS — Managed Key Lifecycle

This example runs encrypted vector search with the **enVector KMS** managing the secret key.

With KMS enabled, the secret key is generated and held inside the KMS trusted boundary — the SDK and the enVector server never see it. Your code still calls an ordinary `search()`; KMS transparently decrypts the encrypted scores on the client's behalf.

The notebook then walks through the key lifecycle — **rotate**, **suspend**, **destroy** — and shows how each transition affects search.

## Prerequisites

Start the stack with the KMS overlay with TLS mode.

```bash
cd ../docker-compose
./start_envector.sh --kms
```

It also brings up the shared local CA, and then export the CA bundle and point the notebook at it.

```bash
# <project> is COMPOSE_PROJECT_NAME from your .env (default: my_project)
docker cp ev-step-ca-1:/certs/ca/root_ca.crt $(pwd)/root_ca.crt
export KMS_SECURE=true KMS_CA_CERT=$(pwd)/root_ca.crt
```

In [ ]:
import os

import numpy as np
import pyenvector as ev
from pyenvector import KMSClient

## Configuration

Set `KMS_SECURE=true` (with `KMS_CA_CERT`) when you started the stack with `--kms` (TLS).
Omit it (or set `KMS_SECURE=false`) for the `--kms-notls` variant.
The `preset`/`eval_mode` pair must match the KMS server configuration; this example uses `ip3`/`mm32`.

In [ ]:
address = os.getenv("ENVECTOR_ADDRESS", "localhost:50050")
# Fetch a Keycloak token in-notebook; `!cmd` captures stdout and the token is the last line.
_out = !../../deployment/scripts/auth/get_keycloak_token.sh app-a password
access_token = _out[-1].strip()

kms_address = os.getenv("KMS_ADDRESS", "localhost:50090")
kms_secure = os.getenv("KMS_SECURE", "false").lower() == "true"
kms_ca_cert = os.getenv("KMS_CA_CERT", "root_ca.crt") if kms_secure else None

key_id = "test_key_mm32"
index_name = "test_index"
dim = 512
preset = "ip3"      # must match the KMS server's configured preset
eval_mode = "mm32"  # must match the KMS server's configured eval mode

## 1. Initialize — KMS manages the secret key

`ev.init(..., kms_address=...)` connects to both the enVector endpoint and the KMS. With `auto_key_setup=True`, the KMS generates the key and registers the public/eval material with the server, while the secret key stays inside KMS.

In [ ]:
ev.init(
    address=address,
    access_token=access_token,
    secure=False,
    kms_address=kms_address,
    kms_secure=kms_secure,
    kms_ca_cert=kms_ca_cert,
    index_name=index_name,
    dim=dim,
    key_id=key_id,
    preset=preset,
    eval_mode=eval_mode,
    auto_key_setup=True,
)

## 2. Create an index and insert data

Vectors are normalized for inner-product similarity, matching the `ip3` preset.

In [ ]:
index = ev.create_index(index_name, dim=dim)

rng = np.random.default_rng(seed=42)
vectors = rng.random((10, dim))
vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)  # normalize for inner product
metadata = [f"item_{i}" for i in range(len(vectors))]

index.insert(vectors, metadata)
query = vectors[0]

## 3. Encrypted search with the managed key

`index.search()` runs exactly as it would without KMS. Under the hood the SDK asks the KMS to decrypt the encrypted scores — the secret key never leaves KMS.

In [ ]:
results = index.search(query, top_k=3, output_fields=["metadata"])[0]
for row in results:
    print(f"id={row['id']} score={row['score']:.6f} metadata={row['metadata']}")

## 4. Inspect the key

Connect a `KMSClient` to drive key-lifecycle operations directly. `get_key_details` returns the version history; each version carries a state such as `KEY_STATE_ACTIVE`.

In [ ]:
kms = KMSClient(
    address=kms_address,
    secure=kms_secure,
    access_token=access_token,
    ca_cert=kms_ca_cert,
)
details = kms.get_key_details(key_id)
print("versions:", [v.get("state") for v in details.get("versions", [])])

## 5. Rotate the key

Rotation adds a new key version. Data inserted under the previous version stays searchable.

In [ ]:
kms.rotate_key(key_id, reason="notebook rotate")

results_after_rotate = index.search(query, top_k=3, output_fields=["metadata"])[0]
print("search after rotate still works:", len(results_after_rotate), "rows")

## 6. Suspend the key

A suspended key cannot decrypt scores, so managed search fails until the key is reactivated (an administrative operation).

In [ ]:
kms.suspend_key(key_id, reason="notebook suspend")

try:
    index.search(query, top_k=3, output_fields=["metadata"])
    print("unexpected: search succeeded while suspended")
except Exception as exc:
    print(f"search failed as expected while suspended: {type(exc).__name__}")

## 7. Destroy the key

Destroying a key is irreversible. Any data encrypted under it becomes permanently unrecoverable.

In [ ]:
kms.destroy_key(key_id, reason="notebook destroy")

try:
    index.search(query, top_k=3, output_fields=["metadata"])
    print("unexpected: search succeeded after destroy")
except Exception as exc:
    print(f"search failed as expected after destroy: {type(exc).__name__}")

## Clean Up

Delete the index. The key is already destroyed above, so `delete_key` is best-effort.

In [ ]:
ev.drop_index(index_name)
try:
    ev.delete_key(key_id)
except Exception as exc:
    print(f"delete_key skipped (key already destroyed): {type(exc).__name__}")